## **Setup**

In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [3]:
if IS_COLAB:
    !pip install optuna

import optuna

In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [13]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_jaccard")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + '_jaccard'

In [32]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "jaccard",
        "topK": optuna_trial.suggest_int("topK", 10, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [33]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-19 23:22:55,378] A new study created in RDB with name: ItemKNNCFRecommender_jaccard


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1869.98 column/sec. Elapsed time 3.73 sec
  Fold 1/5 - Score: 0.19071058221340637
Similarity column 6969 (100.0%), 1890.82 column/sec. Elapsed time 3.69 sec
  Fold 2/5 - Score: 0.19234644410358653
Similarity column 6969 (100.0%), 1885.42 column/sec. Elapsed time 3.70 sec
  Fold 3/5 - Score: 0.19264026408754015
Similarity column 6969 (100.0%), 1899.83 column/sec. Elapsed time 3.67 sec
  Fold 4/5 - Score: 0.19024388274373447
Similarity column 6969 (100.0%), 1884.01 column/sec. Elapsed time 3.70 sec
  Fold 5/5 - Score: 0.19059044444006393
[I 2025-11-19 23:25:21,280] Trial 0 finished with value: 0.19130632351766627 and parameters: {'topK': 487, 'shrink': 494, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 0 with value: 0.19130632351766627.
Similarity column 6969 (100.0%), 1907.87 column/sec. Elapsed time 3.65 sec
  Fold 1/5 - Score: 0.1961123333888447
Similarity column 6969 (100.0%), 1915.00 column/sec. Elapsed time 3.64 sec
  Fold 2/5 - 

In [34]:
optuna.visualization.plot_optimization_history(optuna_study)

In [35]:
optuna.visualization.plot_param_importances(optuna_study)

In [36]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [8]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "jaccard",
        "topK": optuna_trial.suggest_int("topK", 15, 25),
        "shrink": optuna_trial.suggest_int("shrink", 270, 330),
        "normalize": True,
        "feature_weighting": "BM25",
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 1.4, 1.6)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.07, 0.09)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-20 12:21:51,595] A new study created in RDB with name: ItemKNNCFRecommender_jaccard_refined


  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1781.36 column/sec. Elapsed time 3.91 sec
  Fold 1/5 - Score: 0.21519186650617458
Similarity column 6969 (100.0%), 1755.67 column/sec. Elapsed time 3.97 sec
  Fold 2/5 - Score: 0.21628572664616963
Similarity column 6969 (100.0%), 1682.31 column/sec. Elapsed time 4.14 sec
  Fold 3/5 - Score: 0.21595178789514313
Similarity column 6969 (100.0%), 1803.95 column/sec. Elapsed time 3.86 sec
  Fold 4/5 - Score: 0.21513221206455616
Similarity column 6969 (100.0%), 1750.64 column/sec. Elapsed time 3.98 sec
  Fold 5/5 - Score: 0.21714021931630853
[I 2025-11-20 12:23:22,451] Trial 0 finished with value: 0.2159403624856704 and parameters: {'topK': 15, 'shrink': 301, 'BM25_k1': 1.516606566109671, 'BM25_b': 0.07521435118994499}. Best is trial 0 with value: 0.2159403624856704.
Similarity column 6969 (100.0%), 1867.03 column/sec. Elapsed time 3.73 sec
  Fold 1/5 - Score: 0.21349329219976226
Similarity column 6969 (100.0%), 1773.84 column/sec. Elapsed time 3.93 sec
  Fol

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- 1 trial
Best Value: 0.21398300093280462
Best Params: {'topK': 22, 'shrink': 298, 'normalize': True, 'feature_weighting': 'BM25', 'BM25_k1': 1.4997472381457673, 'BM25_b': 0.07837769441098277}
- 2 trial
Best Value: 0.21600991275780096
Best Params: {'topK': 15, 'shrink': 294, 'BM25_k1': 1.4220616855595067, 'BM25_b': 0.07281552215895504}

Best Value: 0.21600991275780096
Best Params: {'topK': 15, 'shrink': 294, 'normalize': True, 'feature_weighting': 'BM25', 'BM25_k1': 1.4220616855595067, 'BM25_b': 0.07281552215895504}
